# Extract transition and reward functions

In [28]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
from value_iteration import value_iteration
from env.user_sim import UserSimEnv
import utils
from env.simulate import visualize_rewards, run_env, visualize_counts

In [29]:
MAX_COUNT = 3
NUM_VALS = 3
NUM_STOCHASTIC_STATES = NUM_VALS**3
NUM_STATES = NUM_STOCHASTIC_STATES * (MAX_COUNT+1)**4 
NUM_ACTIONS = 5

In [30]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
save_folder = 'functions\\2'
filename = 'simulation_results_2.csv'
df = pd.read_csv(data_folder + filename)

action_file = 'normalized_challenges.csv'
action_df = pd.read_csv(data_folder + action_file)


In [31]:
state_features = ['tiredness', 'time_avail', 'pu_state']
reward_signals = ['obs_time', 'obs_liked', 'obs_pu']

expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = action_df[expert_score_cols].values

category_mapping = {"acceptance": 0, "distraction": 1, "problem_solving": 2, "social_support": 3}

action_df['category_id'] = action_df['category'].map(category_mapping)
action_categories = action_df['category_id'].values

df = df.merge(action_df[['action_id', 'category_id'] + expert_score_cols], on='action_id', how='left')

NUM_ACTIONS = len(action_df)

df.head()

,user_id,timestep,action_id,completed,rw_state,time_avail,motivation,tiredness,obs_diff,pu_state,obs_time,obs_liked,obs_pu,category_id,score_acceptance,score_distraction,score_problem_solving,score_social_support
0,1,1,44,True,1.0,3.0,4.0,3.0,3.0,4.0,5.0,4.0,5.0,1,0.0,1.000000,0.333333,0.000000
1,1,2,92,False,1.0,3.0,5.0,5.0,NaN,5.0,NaN,NaN,NaN,3,0.0,0.000000,0.000000,0.666667
2,1,3,101,True,1.0,4.0,5.0,4.0,6.0,5.0,3.0,7.0,7.0,3,0.0,0.333333,0.000000,0.666667
3,1,4,46,True,2.0,4.0,6.0,4.0,4.0,6.0,4.0,4.0,7.0,1,0.0,0.666667,0.000000,0.333333
4,1,5,79,False,2.0,5.0,7.0,3.0,NaN,7.0,NaN,NaN,NaN,3,0.0,0.333333,0.000000,0.666667


### Extract states

In [32]:
def get_feature_representation(feature_vals, num_vals=NUM_VALS):
    # Bin the feature values into discrete categories
    # Maybe use qcute for relative binning?
    binned_vals = pd.cut(feature_vals, bins=num_vals, labels=False)
    return binned_vals

In [33]:
for feature in state_features:
    df['s_' + feature] = get_feature_representation(df[feature])

state_cols = ['s_' + feature for feature in state_features]

df['s_next'] = df.groupby('user_id')[state_cols].shift(-1).values.tolist()
df['s_current'] = df[state_cols].values.tolist()

# drop rows where s_next contains NaN values (last row of each user)
df = df[~df['s_next'].apply(lambda x: any(pd.isna(i) for i in x))]
df = df.drop(state_cols + state_features, axis=1)

#### Extract counts per category

In [34]:
for c in range(4):
    df[f"cat_{c}"] = ((df["category_id"] == c) & df["completed"]).astype(int)

cat_cols = [f"cat_{c}" for c in range(4)]

df[cat_cols] = df.groupby("user_id")[cat_cols].cumsum()
df["count"] = df[cat_cols].values.tolist()

df['s_count_next'] = df['count'].apply(lambda x: [min(count - min(x), MAX_COUNT) for count in x])
df['s_count'] = df.groupby('user_id')['s_count_next'].shift(1, fill_value=[0]*4)

df.drop(cat_cols, axis=1, inplace=True)

#### Extract skill level per category

In [35]:
def map_to_tier(normalized_score):
    """Maps a 0.0-1.0 progress value to discrete skill tiers."""
    if normalized_score >= 0.75: return 1.0
    if normalized_score >= 0.50: return 0.67
    if normalized_score >= 0.25: return 0.33
    return 0.0


for col in expert_score_cols:
    raw_sum_col = f'raw_sum_{col}'
    df[raw_sum_col] = df.groupby('user_id')[col].transform(lambda x: x.shift(1, fill_value=0).cumsum())
    df[f'skill_{col}'] = df[raw_sum_col].apply(map_to_tier)

skill_cols = [f'skill_{col}' for col in expert_score_cols]
df['skill_current'] = df[skill_cols].values.tolist()
df['skill_next'] = df.groupby('user_id')[skill_cols].shift(-1).values.tolist()

df = df.drop(columns=[f'raw_sum_{col}' for col in expert_score_cols])

#### Prepare reward signals

In [36]:
# reward for skill improvement = reward for completing the action * (sum of skill tiers after - sum of skill tiers before) / max possible increase in skill tiers
df['r_skill'] = df['completed'] * (df['skill_next'].apply(lambda x: sum(x)) - df['skill_current'].apply(lambda x: sum(x))) / 4
df['r_expert'] = df[expert_score_cols].sum(axis=1) / 4  # alternative reward based on expert scores for the action
df['r_diversity'] = df['completed'] * (1 - (1 / (MAX_COUNT + 1)) * df.apply(lambda row: row['s_count'][action_categories[row['action_id']]], axis=1))
df[reward_signals] = df[reward_signals].div(7)  # Normalize rewards to [0, 1] range

df.head()


,user_id,timestep,action_id,completed,rw_state,motivation,obs_diff,obs_time,obs_liked,obs_pu,...,s_count,skill_score_acceptance,skill_score_distraction,skill_score_problem_solving,skill_score_social_support,skill_current,skill_next,r_skill,r_expert,r_diversity
0,1,1,44,True,1.0,4.0,3.0,0.714286,0.571429,0.714286,...,"[0, 0, 0, 0]",0.0,0.0,0.00,0.00,"[0.0, 0.0, 0.0, 0.0]","[0.0, 1.0, 0.33, 0.0]",0.3325,0.333333,1.00
1,1,2,92,False,1.0,5.0,NaN,NaN,NaN,NaN,...,"[0, 1, 0, 0]",0.0,1.0,0.33,0.00,"[0.0, 1.0, 0.33, 0.0]","[0.0, 1.0, 0.33, 0.67]",0.0000,0.166667,0.00
2,1,3,101,True,1.0,5.0,6.0,0.428571,1.000000,1.000000,...,"[0, 1, 0, 0]",0.0,1.0,0.33,0.67,"[0.0, 1.0, 0.33, 0.67]","[0.0, 1.0, 0.33, 1.0]",0.0825,0.250000,1.00
3,1,4,46,True,2.0,6.0,4.0,0.571429,0.571429,1.000000,...,"[0, 1, 0, 1]",0.0,1.0,0.33,1.00,"[0.0, 1.0, 0.33, 1.0]","[0.0, 1.0, 0.33, 1.0]",0.0000,0.250000,0.75
4,1,5,79,False,2.0,7.0,NaN,NaN,NaN,NaN,...,"[0, 2, 0, 1]",0.0,1.0,0.33,1.00,"[0.0, 1.0, 0.33, 1.0]","[0.0, 1.0, 0.33, 1.0]",0.0000,0.250000,0.00


### Learn reward functions

In [37]:
df['s_idx'] = df['s_current'].apply(lambda s: utils.state_to_idx(tuple(s), max_count=0))
df['sp_idx'] = df['s_next'].apply(lambda sp: utils.state_to_idx(tuple(sp), max_count=0))
df['s_full_idx'] = df.apply(lambda row: utils.state_to_idx(tuple(row['s_current']) + tuple(row['s_count']), max_count=MAX_COUNT), axis=1)

# Learn reward functions
# reward is given by R(s, a) = E[reward | s, a] -> do i want it probabilistic or deterministic? maybe start with deterministic (mean reward) and then move to probabilistic (reward distribution)
# def get_reward_probs(df, reward_col):
#     counts = df.groupby(['s_idx', 'action_id', reward_col]).size().unstack(fill_value=0)
#     return counts.div(counts.sum(axis=1), axis=0)

# P_obs_time = get_reward_probs(df, 'obs_time')
# P_obs_liked = get_reward_probs(df, 'obs_liked')
# P_obs_pu = get_reward_probs(df, 'obs_pu')


# Mean reward for each (s, a) pair
obs_cols = ['obs_time', 'obs_liked', 'obs_pu', 'r_expert', 'r_diversity', 'completed']

skill_levels = [0.0, 0.33, 0.67, 1.0]
all_skill_levels = [[s1, s2, s3, s4] for s1 in skill_levels for s2 in skill_levels for s3 in skill_levels for s4 in skill_levels]
def skill_to_idx(skill_list):
    return all_skill_levels.index(skill_list)
num_skill_states = len(all_skill_levels)

# df['skill_idx'] = df['skill_current'].apply(skill_to_idx)
# df['skill_next_idx'] = df['skill_next'].apply(skill_to_idx)

def get_reward_matrix(df, obs_cols, skills=False):
    if skills:
        reward_lookup = df.fillna(value=0).groupby(['s_full_idx', 'skill_idx', 'action_id'])[obs_cols].mean()
        reward_matrix = np.zeros((NUM_STATES, num_skill_states, NUM_ACTIONS, len(obs_cols)))
        for (s, sk, a), rewards_row in reward_lookup.iterrows():
            reward_matrix[s, sk, a, :] = rewards_row.values
        # fill missing rewards with values for similar states (same state, different skill levels)
        for s in range(NUM_STATES):
            for a in range(NUM_ACTIONS):
                for sk in range(num_skill_states):
                    if np.all(reward_matrix[s, sk, a, :] == 0):
                        # find other skill levels for the same state and action
                        similar_rewards = reward_matrix[s, :, a, :]
                        # average the rewards from similar states
                        reward_matrix[s, sk, a, :] = similar_rewards.mean(axis=0)
        return reward_matrix
    else:
        reward_lookup = df.fillna(value=0).groupby(['s_full_idx', 'action_id'])[obs_cols].mean()
        reward_matrix = np.zeros((NUM_STATES, NUM_ACTIONS, len(obs_cols)))
        for (s, a), rewards_row in reward_lookup.iterrows():
            reward_matrix[s, a, :] = rewards_row.values
        return reward_matrix
    
filename = 'reward_matrix.npy'
file_path = os.path.join(data_folder, save_folder, filename)
if not os.path.exists(file_path):
    print("Reward matrix not found, creating reward matrix...")
    reward_matrix = get_reward_matrix(df, obs_cols)
    # save reward matrix for use in environment
    np.save(file_path, reward_matrix)
else:
    print("Reward matrix found, loading reward matrix...")
    reward_matrix = np.load(file_path)

Reward matrix found, loading reward matrix...


### Learn transition function
Our state consists of a stochastic part: tiredness, time available, and motivation. We thus need to learn the transition probabilities for these states and the actions.

$T(s,a,s') = \frac{N(s,a,s')}{N(s,a)}$ 

In [38]:
file_path = os.path.join(data_folder, save_folder, 'transition_probs_all_actions.npy')
if not os.path.exists(file_path):
    print("Transition matrix not found, creating transition matrix...")
    # transition probabilities for (tiredness, time available, perceived usefulness)
    transition_counts = df.groupby(['s_idx', 'action_id', 'sp_idx']).size().unstack(fill_value=0)

    # Normalize to get probabilities
    transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)

    # convert to numpy array
    transition_probs_array = np.zeros((NUM_STOCHASTIC_STATES, NUM_ACTIONS, NUM_STOCHASTIC_STATES))

    for (s, a), s_next_col in transition_probs.iterrows():
        for s_next, prob in s_next_col.items():
            if pd.notna(s_next):
                transition_probs_array[s, a, int(s_next)] = prob

    np.save(file_path, transition_probs_array)
else:
    print("Transition matrix found, loading transition matrix...")
    transition_probs_array = np.load(file_path)

# for s in range(NUM_STATES):
#     state = utils.idx_to_state(s, max_count=MAX_COUNT)
#     # state is (tiredness, time_avail, pu_state, acceptance_count, distraction_count, problemsolving_count, socialsupport_count)
#     tir, time, pu, acc, dis, ps, ss = state
#     for a in range(NUM_ACTIONS):
#         prob_done = reward_matrix[s, a, obs_cols.index('completed')]

# # fill missing transitions with uniform probabilities
# for s in range(NUM_STATES):
#     for a in range(NUM_ACTIONS):
#         if np.all(transition_probs_array[s, a, :] == 0):
#             transition_probs_array[s, a, :] = 1 / NUM_STATES

Transition matrix found, loading transition matrix...


In [39]:
file_path = os.path.join(data_folder, save_folder, 'completion_probs.npy')
if not os.path.exists(file_path):
    print("Completion matrix not found, creating completion matrix...")
    # completion probabilities for each (s, a) pair
    completion_probs = df.groupby(['s_full_idx', 'action_id'])['completed'].mean()
    completion_probs_array = np.zeros((NUM_STATES, NUM_ACTIONS))
    for s in range(NUM_STATES):
        for a in range(NUM_ACTIONS):
            if (s, a) in completion_probs:
                completion_probs_array[s, a] = completion_probs[s, a]

else:
    print("Completion matrix found, loading completion matrix...")
    completion_probs_array = np.load(file_path)

Completion matrix not found, creating completion matrix...


In [42]:
np.save(file_path, completion_probs_array)

In [45]:
print(np.all(reward_matrix[:, :, -1] == completion_probs_array))


True


In [13]:
def run_env(env, num_simulations, policy=None, verbose=False):
    rewards_list = []
    counts_list = []
    for _ in range(num_simulations):
        obs, _ = env.reset()
        done = False
        total_rewards = []
        total_counts = []
        while not done:
            state = utils.state_to_idx((obs[0], obs[1], obs[2]), max_count=0)
            action = policy[state] if policy is not None else env.action_space.sample()
            obs, rewards, terminated, truncated, info = env.step(action)
            if verbose:
                print(f"Action: {action}, Rewards: {rewards}, Info: {info}")
            total_rewards.append(rewards)
            total_counts.append(obs[3:])
            done = terminated or truncated
        rewards_list.append(total_rewards)
        counts_list.append(total_counts)
    return rewards_list, counts_list

In [41]:
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = 5, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs_array, 
                 completion_probs=completion_probs_array,
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)
# rewards, counts = run_env(env, 500)
# visualize_rewards(rewards)
# visualize_counts(counts)

# V, policy = value_iteration(env)
# print("Optimal policy:")
# print(policy)
# rewards = run_env(env, 50, policy=policy)
# visualize_rewards(rewards)
